# Promoter and Regulatory Sequence Analysis

**Tier 3 -- Applied Bioinformatics**

Gene expression is orchestrated by regulatory elements embedded in DNA. This notebook covers how to identify and analyze these elements computationally: promoter regions, CpG islands, transcription factor binding sites (TFBS), and methylation patterns.

**Prerequisites:** Tier 2 (BioPython basics, sequence handling, basic statistics)  
**Libraries:** `numpy`, `pandas`, `matplotlib`, `re`

---

[← Previous: Microbial Diversity Analysis](../../Tier_3_Applied_Bioinformatics/04_Microbial_Diversity/01_microbial_diversity.ipynb) | [Next: Statistics for Bioinformatics →](../../Tier_3_Applied_Bioinformatics/06_Statistics_for_Bioinformatics/01_statistics_for_bioinformatics.ipynb)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re
from collections import Counter

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 12

---
## 1. Gene Regulation: The Big Picture

Every cell in an organism contains the same DNA, yet a neuron and a liver cell behave very differently. The key lies in **gene regulation** -- controlling which genes are transcribed, when, and how much.

### Key regulatory elements

| Element | Location | Function |
|---------|----------|----------|
| **Promoter** | Immediately upstream of gene (typically -1 to -1000 bp from TSS) | Recruits RNA polymerase, initiates transcription |
| **Enhancer** | Can be thousands of bp away (upstream, downstream, or within introns) | Boosts transcription when bound by activators |
| **Silencer** | Variable distance from gene | Represses transcription |
| **Insulator** | Between regulatory elements | Blocks enhancer-promoter interactions |

### Transcription factors (TFs)

Transcription factors are proteins that bind specific short DNA motifs (6-20 bp) to regulate gene expression. They can be:
- **Activators** -- enhance transcription by recruiting RNA polymerase
- **Repressors** -- block transcription

The **transcription start site (TSS)** is position +1, where RNA polymerase begins transcribing. Positions upstream are negative (e.g., -30 for the TATA box), and downstream positions are positive.

---
## 2. Promoter Elements

### 2.1 The TATA box

The TATA box is a core promoter element found approximately 25-35 bp upstream of the TSS. Its consensus sequence is **TATAAA**, though variations exist (TATAWAW where W = A or T).

- Present in roughly 10-20% of human genes (more common in highly regulated, tissue-specific genes)
- Bound by the TATA-binding protein (TBP), part of the TFIID complex
- TATA-less promoters often rely on other elements like the Initiator (Inr) or DPE

In [ ]:
def find_tata_boxes(sequence, strict=True):
    """Find TATA box motifs in a DNA sequence.
    
    strict=True: exact TATAAA
    strict=False: TATA[AT]A[AT] (IUPAC TATAWAW)
    """
    sequence = sequence.upper()
    pattern = 'TATAAA' if strict else 'TATA[AT]A[AT]'
    matches = [(m.start(), m.group()) for m in re.finditer(pattern, sequence)]
    return matches

# Example: a synthetic promoter region (1000 bp upstream of TSS at position 1000)
np.random.seed(42)
bases = 'ACGT'
# Generate random sequence, then insert a TATA box near expected position
random_seq = ''.join(np.random.choice(list(bases), size=2000))
# Insert TATA box at position 970 (i.e., -30 relative to TSS at 1000)
promoter_seq = random_seq[:970] + 'TATAAAG' + random_seq[977:]

hits = find_tata_boxes(promoter_seq, strict=True)
print(f"TATA box hits in 2000 bp region (TSS at position 1000):")
for pos, motif in hits:
    relative = pos - 1000
    print(f"  Position {pos} (TSS{relative:+d}): {motif}")

### 2.2 CpG islands

**CpG dinucleotides** (cytosine followed by guanine, written CpG to distinguish from C-G base pairing) are statistically underrepresented in vertebrate genomes because methylated cytosines in CpG context tend to mutate to thymine over evolutionary time.

However, clusters of CpGs called **CpG islands** are preserved near promoters of ~70% of human genes. They are defined by:

1. **Length** >= 200 bp
2. **GC content** >= 50%
3. **Observed/Expected CpG ratio** >= 0.6

The observed/expected ratio is:

$$\text{CpG O/E} = \frac{N_{CpG} \times L}{N_C \times N_G}$$

where $N_{CpG}$ is the count of CpG dinucleotides, $L$ is the window length, and $N_C$, $N_G$ are counts of C and G nucleotides.

In [ ]:
def cpg_island_scanner(sequence, window=200, step=1, gc_thresh=0.5, oe_thresh=0.6):
    """Scan a sequence for CpG islands using a sliding window.
    
    Returns a list of (start, end, gc_content, cpg_oe_ratio) for qualifying windows.
    """
    sequence = sequence.upper()
    islands = []
    
    for i in range(0, len(sequence) - window + 1, step):
        win = sequence[i:i + window]
        n_c = win.count('C')
        n_g = win.count('G')
        n_cpg = win.count('CG')  # CpG dinucleotide count
        gc_content = (n_c + n_g) / window
        
        # Avoid division by zero
        if n_c > 0 and n_g > 0:
            cpg_oe = (n_cpg * window) / (n_c * n_g)
        else:
            cpg_oe = 0.0
        
        if gc_content >= gc_thresh and cpg_oe >= oe_thresh:
            islands.append((i, i + window, gc_content, cpg_oe))
    
    return islands

print("CpG island scanner defined. We will use it in the practical section below.")

### 2.3 Transcription factor binding sites (TFBS)

TFs recognize short, degenerate DNA motifs. These are represented as **position weight matrices (PWMs)** or **position frequency matrices (PFMs)**.

A PWM stores the log-likelihood of each base at each position of the motif relative to background:

$$w_{b,i} = \log_2\frac{p(b, i)}{p_{bg}(b)}$$

where $p(b, i)$ is the probability of base $b$ at position $i$ and $p_{bg}(b)$ is the background probability.

### Key databases

| Database | Description | Access |
|----------|-------------|--------|
| **JASPAR** | Open-access, curated TF binding profiles | https://jaspar.elixir.no |
| **TRANSFAC** | Comprehensive commercial/academic TF database | https://genexplain.com/transfac |
| **HOCOMOCO** | Human and mouse TF motifs from ChIP-seq | https://hocomoco11.autosome.org |

In the source data for this course, TFBS were identified using TRANSFAC matrices aligned to rice promoter regions. Each position in a gene's promoter was annotated with the number of TFBS overlapping it.

In [ ]:
# Build a simple PWM for a TATA box motif from example binding sites
tata_sites = [
    'TATAAAG',
    'TATAAAT',
    'TATAAAA',
    'TATATAG',
    'TATAAAC',
    'TATAAAT',
    'TATAAAG',
    'TATAAAT',
    'TATAAAA',
    'TATAAAG',
]

def build_pfm(sites):
    """Build a position frequency matrix from aligned binding sites."""
    length = len(sites[0])
    pfm = {b: [0] * length for b in 'ACGT'}
    for site in sites:
        for i, base in enumerate(site.upper()):
            pfm[base][i] += 1
    return pfm

def pfm_to_pwm(pfm, pseudocount=0.5, bg=None):
    """Convert PFM to PWM (log-odds scores)."""
    if bg is None:
        bg = {'A': 0.25, 'C': 0.25, 'G': 0.25, 'T': 0.25}
    n = sum(pfm[b][0] for b in 'ACGT')  # total sequences
    length = len(pfm['A'])
    pwm = {b: [0.0] * length for b in 'ACGT'}
    for b in 'ACGT':
        for i in range(length):
            freq = (pfm[b][i] + pseudocount) / (n + 4 * pseudocount)
            pwm[b][i] = np.log2(freq / bg[b])
    return pwm

pfm = build_pfm(tata_sites)
pwm = pfm_to_pwm(pfm)

print("Position Frequency Matrix (PFM):")
pfm_df = pd.DataFrame(pfm, index=[f"pos {i+1}" for i in range(len(pfm['A']))])
print(pfm_df)
print("\nPosition Weight Matrix (PWM, log2 odds):")
pwm_df = pd.DataFrame(pwm, index=[f"pos {i+1}" for i in range(len(pwm['A']))])
print(pwm_df.round(3))

In [ ]:
def score_sequence(sequence, pwm):
    """Score a sequence against a PWM. Returns the sum of log-odds."""
    score = 0.0
    for i, base in enumerate(sequence.upper()):
        if base in pwm:
            score += pwm[base][i]
    return score

def scan_with_pwm(sequence, pwm, threshold=0.0):
    """Scan a sequence with a PWM and return hits above threshold."""
    motif_len = len(pwm['A'])
    hits = []
    for i in range(len(sequence) - motif_len + 1):
        subseq = sequence[i:i + motif_len]
        sc = score_sequence(subseq, pwm)
        if sc >= threshold:
            hits.append((i, subseq, sc))
    return hits

# Scan our synthetic promoter with the TATA PWM
hits = scan_with_pwm(promoter_seq, pwm, threshold=5.0)
print(f"PWM hits (score >= 5.0) in synthetic promoter:")
for pos, seq, sc in sorted(hits, key=lambda x: -x[2])[:10]:
    relative = pos - 1000
    print(f"  Position {pos} (TSS{relative:+d}): {seq}  score={sc:.2f}")

---
## 3. TSS Prediction Concepts

Identifying the **transcription start site (TSS)** is fundamental for promoter analysis. Computational approaches include:

1. **Signal-based methods**: Look for known motifs (TATA box at -25 to -30, Inr at +1, DPE at +30)
2. **Content-based methods**: CpG island presence, GC content profiles
3. **Experimental data integration**: CAGE (Cap Analysis of Gene Expression), which captures 5' ends of mRNAs
4. **Machine learning**: Combine multiple features (motifs, chromatin marks, conservation) to predict TSS

In the rice promoter dataset used in this course, TSS positions were known, and each gene had 1000 bp upstream and 1000 bp downstream of the TSS. This allows us to study the distribution of regulatory signals relative to TSS.

In [ ]:
# Simulate signal distribution around TSS (based on patterns from the rice promoter study)
positions = np.arange(-1000, 1000)

# TATA box: peak around -30
tata_signal = np.exp(-0.5 * ((positions + 30) / 5) ** 2) * 50
tata_signal += np.random.poisson(1, len(positions))

# CpG density: enriched around TSS
cpg_signal = np.exp(-0.5 * (positions / 200) ** 2) * 30
cpg_signal += np.random.poisson(2, len(positions))

# TFBS density: enriched upstream
tfbs_signal = np.where(positions < 0, 15 + 10 * np.exp(-0.5 * ((positions + 200) / 150) ** 2), 5)
tfbs_signal += np.random.poisson(2, len(positions))

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].fill_between(positions, tata_signal, alpha=0.5, color='orange')
axes[0].axvline(0, color='red', linestyle='--', label='TSS')
axes[0].set_ylabel('TATA box signal')
axes[0].set_title('Regulatory Signal Distribution Around the TSS')
axes[0].legend()

axes[1].fill_between(positions, cpg_signal, alpha=0.5, color='blue')
axes[1].axvline(0, color='red', linestyle='--', label='TSS')
axes[1].set_ylabel('CpG density')
axes[1].legend()

axes[2].fill_between(positions, tfbs_signal, alpha=0.5, color='green')
axes[2].axvline(0, color='red', linestyle='--', label='TSS')
axes[2].set_ylabel('TFBS density')
axes[2].set_xlabel('Position relative to TSS (bp)')
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 4. CpG Island Detection

Let us build a complete CpG island detection pipeline. We will:
1. Generate a realistic promoter sequence with an embedded CpG island
2. Scan it with our sliding window algorithm
3. Merge overlapping windows into island calls
4. Visualize the results

In [ ]:
def generate_promoter_with_cpg_island(length=3000, island_start=1200, island_length=400):
    """Generate a synthetic promoter with an embedded CpG island."""
    np.random.seed(123)
    
    # Background: AT-rich, CpG-depleted (typical vertebrate genome)
    bg_weights = [0.29, 0.21, 0.21, 0.29]  # A, C, G, T
    bg_seq = list(np.random.choice(list('ACGT'), size=length, p=bg_weights))
    
    # CpG island: GC-rich, CpG-enriched
    island_weights = [0.18, 0.32, 0.32, 0.18]
    island_seq = list(np.random.choice(list('ACGT'), size=island_length, p=island_weights))
    
    # Further enrich CpGs in the island
    for i in range(0, island_length - 1, 3):
        if np.random.random() < 0.3:
            island_seq[i] = 'C'
            island_seq[i + 1] = 'G'
    
    # Insert island into background
    bg_seq[island_start:island_start + island_length] = island_seq
    return ''.join(bg_seq)

synth_promoter = generate_promoter_with_cpg_island()
print(f"Synthetic promoter length: {len(synth_promoter)} bp")
print(f"Expected CpG island: positions 1200-1600")

In [ ]:
def merge_island_windows(windows, max_gap=0):
    """Merge overlapping or adjacent CpG island windows into contiguous islands."""
    if not windows:
        return []
    
    sorted_wins = sorted(windows, key=lambda x: x[0])
    merged = [list(sorted_wins[0])]
    
    for start, end, gc, oe in sorted_wins[1:]:
        if start <= merged[-1][1] + max_gap:
            merged[-1][1] = max(merged[-1][1], end)
            merged[-1][2] = max(merged[-1][2], gc)   # keep max GC
            merged[-1][3] = max(merged[-1][3], oe)   # keep max O/E
        else:
            merged.append([start, end, gc, oe])
    
    return merged

# Scan the synthetic promoter
raw_islands = cpg_island_scanner(synth_promoter, window=200, step=10)
merged = merge_island_windows(raw_islands, max_gap=50)

print(f"Raw qualifying windows: {len(raw_islands)}")
print(f"Merged CpG islands:")
for start, end, gc, oe in merged:
    print(f"  {start}-{end} ({end - start} bp)  GC={gc:.2f}  CpG O/E={oe:.2f}")

In [ ]:
# Visualize GC content and CpG O/E ratio along the sequence
def sliding_window_stats(sequence, window=200, step=10):
    """Compute GC content and CpG O/E ratio in sliding windows."""
    gc_vals = []
    oe_vals = []
    midpoints = []
    
    for i in range(0, len(sequence) - window + 1, step):
        win = sequence[i:i + window].upper()
        n_c = win.count('C')
        n_g = win.count('G')
        n_cpg = win.count('CG')
        gc = (n_c + n_g) / window
        oe = (n_cpg * window) / (n_c * n_g) if (n_c > 0 and n_g > 0) else 0
        
        gc_vals.append(gc)
        oe_vals.append(oe)
        midpoints.append(i + window // 2)
    
    return np.array(midpoints), np.array(gc_vals), np.array(oe_vals)

mid, gc, oe = sliding_window_stats(synth_promoter)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(mid, gc, color='steelblue', linewidth=1)
ax1.axhline(0.5, color='red', linestyle='--', alpha=0.7, label='GC >= 50% threshold')
for s, e, _, _ in merged:
    ax1.axvspan(s, e, alpha=0.2, color='green', label='CpG island' if s == merged[0][0] else '')
ax1.set_ylabel('GC content')
ax1.set_title('CpG Island Detection in Synthetic Promoter')
ax1.legend()

ax2.plot(mid, oe, color='darkorange', linewidth=1)
ax2.axhline(0.6, color='red', linestyle='--', alpha=0.7, label='CpG O/E >= 0.6 threshold')
for s, e, _, _ in merged:
    ax2.axvspan(s, e, alpha=0.2, color='green')
ax2.set_ylabel('CpG observed/expected')
ax2.set_xlabel('Position (bp)')
ax2.legend()

plt.tight_layout()
plt.show()

---
## 5. DNA Methylation and Gene Regulation

### 5.1 Methylation basics

**DNA methylation** is the addition of a methyl group to the 5-carbon of cytosine, primarily at CpG dinucleotides. It is a critical epigenetic modification:

- **Methylated CpG islands** in promoters are associated with gene **silencing**
- **Unmethylated CpG islands** in promoters are associated with **active** transcription
- Methylation patterns are heritable through cell division (maintained by DNMT1)

### 5.2 Methylation in the rice promoter dataset

In the rice promoter dataset, methylation data was encoded per position:
- **0** = not a CpG site
- **+1** = methylated CpG
- **-1** = unmethylated CpG

This allows us to study the spatial relationship between methylation and the TSS.

In [ ]:
# Simulate methylation landscape around TSS (inspired by the rice dataset)
np.random.seed(42)
n_genes = 500
region_size = 2000  # -1000 to +1000 around TSS
positions = np.arange(-1000, 1000)

# Methylation drops sharply near TSS (typical pattern in active genes)
meth_prob = 0.3 + 0.5 * (1 / (1 + np.exp(-0.01 * (np.abs(positions) - 300))))
# CpG density is higher near TSS
cpg_prob = 0.05 + 0.15 * np.exp(-0.5 * (positions / 300) ** 2)

meth_sum = np.zeros(region_size)
cpg_count = np.zeros(region_size)

for _ in range(n_genes):
    is_cpg = np.random.random(region_size) < cpg_prob
    is_methylated = np.random.random(region_size) < meth_prob
    # +1 for methylated CpG, -1 for unmethylated CpG, 0 for non-CpG
    meth_values = np.where(is_cpg, np.where(is_methylated, 1, -1), 0)
    meth_sum += meth_values
    cpg_count += is_cpg.astype(int)

# Smooth with a rolling average
window = 50
meth_smooth = pd.Series(meth_sum).rolling(window, center=True).mean()
cpg_smooth = pd.Series(cpg_count).rolling(window, center=True).mean()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(positions, cpg_smooth, color='blue', linewidth=1.5)
ax1.axvline(0, color='red', linestyle='--', alpha=0.7, label='TSS')
ax1.set_ylabel('Mean CpG count per gene')
ax1.set_title('CpG Density and Methylation Around TSS (simulated, n=500 genes)')
ax1.legend()

ax2.plot(positions, meth_smooth, color='purple', linewidth=1.5)
ax2.axvline(0, color='red', linestyle='--', alpha=0.7, label='TSS')
ax2.axhline(0, color='gray', linestyle=':', alpha=0.5)
ax2.set_ylabel('Net methylation signal\n(positive = more methylated)')
ax2.set_xlabel('Position relative to TSS (bp)')
ax2.legend()

plt.tight_layout()
plt.show()

print("Observation: CpG density peaks near TSS, but net methylation dips there.")
print("This reflects the classic 'methylation valley' at active promoters.")

---
## 6. Promoter Sequence Analysis

### 6.1 GC content profiles

GC content is not uniform across genomes. Promoter regions, especially those with CpG islands, tend to have higher GC content. We can compute **GC content profiles** across many genes to find average patterns.

In [ ]:
def generate_promoter_collection(n_genes=200, length=2000, tss_pos=1000):
    """Generate a collection of synthetic promoter sequences with realistic features."""
    np.random.seed(99)
    promoters = []
    
    for _ in range(n_genes):
        seq = []
        for pos in range(length):
            rel = pos - tss_pos
            # Higher GC near TSS
            gc_bias = 0.25 + 0.20 * np.exp(-0.5 * (rel / 200) ** 2)
            p_c = gc_bias / 2
            p_g = gc_bias / 2
            p_a = (1 - gc_bias) / 2
            p_t = (1 - gc_bias) / 2
            base = np.random.choice(list('ACGT'), p=[p_a, p_c, p_g, p_t])
            seq.append(base)
        promoters.append(''.join(seq))
    
    return promoters

promoters = generate_promoter_collection()
print(f"Generated {len(promoters)} synthetic promoter sequences of {len(promoters[0])} bp each.")

In [ ]:
# Compute average GC content profile across all promoters
def compute_gc_profile(promoters, window=100, step=5):
    """Compute average GC content at each position across a set of sequences."""
    length = len(promoters[0])
    midpoints = list(range(window // 2, length - window // 2, step))
    profiles = []
    
    for seq in promoters:
        gc = []
        for mid in midpoints:
            start = mid - window // 2
            end = mid + window // 2
            win = seq[start:end]
            gc.append((win.count('C') + win.count('G')) / window)
        profiles.append(gc)
    
    return np.array(midpoints), np.array(profiles)

midpoints, gc_profiles = compute_gc_profile(promoters)
tss_pos = 1000
relative_pos = midpoints - tss_pos

mean_gc = gc_profiles.mean(axis=0)
std_gc = gc_profiles.std(axis=0)

plt.figure(figsize=(14, 5))
plt.plot(relative_pos, mean_gc, color='steelblue', linewidth=2, label='Mean GC content')
plt.fill_between(relative_pos, mean_gc - std_gc, mean_gc + std_gc, alpha=0.2, color='steelblue')
plt.axvline(0, color='red', linestyle='--', label='TSS', linewidth=2)
plt.xlabel('Position relative to TSS (bp)')
plt.ylabel('GC content')
plt.title('Average GC Content Profile Across 200 Promoters')
plt.legend()
plt.tight_layout()
plt.show()

### 6.2 Dinucleotide frequencies

Beyond single-nucleotide composition, **dinucleotide frequencies** reveal important biological signals. For instance:
- **CpG** frequency flags CpG islands
- **CA/TG** dinucleotides relate to chromatin accessibility
- **AA/TT** runs affect DNA bendability (important for nucleosome positioning)

In [ ]:
def compute_dinucleotide_profile(promoters, dinucleotide, window=100, step=10):
    """Compute average frequency of a specific dinucleotide across positions."""
    length = len(promoters[0])
    midpoints = list(range(window // 2, length - window // 2, step))
    profiles = []
    
    for seq in promoters:
        freqs = []
        for mid in midpoints:
            start = mid - window // 2
            end = mid + window // 2
            win = seq[start:end].upper()
            count = sum(1 for i in range(len(win) - 1) if win[i:i+2] == dinucleotide)
            freqs.append(count / (window - 1))
        profiles.append(freqs)
    
    return np.array(midpoints), np.array(profiles)

# Compare CG, CA, and TA dinucleotide profiles
dinucs = ['CG', 'CA', 'TA']
colors = ['blue', 'green', 'orange']

fig, ax = plt.subplots(figsize=(14, 5))

for dinuc, color in zip(dinucs, colors):
    mid, profiles = compute_dinucleotide_profile(promoters, dinuc)
    mean_freq = profiles.mean(axis=0)
    ax.plot(mid - tss_pos, mean_freq, color=color, linewidth=1.5, label=dinuc)

ax.axvline(0, color='red', linestyle='--', label='TSS')
ax.set_xlabel('Position relative to TSS (bp)')
ax.set_ylabel('Dinucleotide frequency')
ax.set_title('Dinucleotide Frequency Profiles Around the TSS')
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Practical: Complete Promoter Analysis Pipeline

Let us put everything together: analyze a set of promoter sequences for TATA boxes, CpG islands, TFBS density, and GC landscape.

In [ ]:
# Generate a more realistic promoter with multiple features
np.random.seed(2024)

def create_annotated_promoter(length=2000, tss=1000):
    """Create a promoter with embedded biological features."""
    # Base sequence
    weights = [0.29, 0.21, 0.21, 0.29]
    seq = list(np.random.choice(list('ACGT'), size=length, p=weights))
    
    # Insert TATA box at -30
    tata_pos = tss - 30
    for i, b in enumerate('TATAAAG'):
        seq[tata_pos + i] = b
    
    # Insert CpG-rich region from -500 to -100 (CpG island)
    for i in range(tss - 500, tss - 100):
        if np.random.random() < 0.35:
            seq[i] = 'C'
            if i + 1 < length:
                seq[i + 1] = 'G'
    
    # Insert a few TFBS-like motifs upstream
    # GATA motif (GATA TF binding)
    for pos in [tss - 200, tss - 350, tss - 450]:
        for i, b in enumerate('AGATAA'):
            seq[pos + i] = b
    
    return ''.join(seq)

test_promoter = create_annotated_promoter()
tss = 1000
print(f"Annotated promoter: {len(test_promoter)} bp")
print(f"Region around TATA box (TSS-35 to TSS-20): {test_promoter[tss-35:tss-20]}")

In [ ]:
# Comprehensive analysis
tss = 1000

# 1. Find TATA boxes
tata_hits = find_tata_boxes(test_promoter, strict=True)
print("=== TATA Box Analysis ===")
for pos, motif in tata_hits:
    print(f"  Found '{motif}' at position {pos} (TSS{pos-tss:+d})")

# 2. Find CpG islands
cpg_raw = cpg_island_scanner(test_promoter, window=200, step=10)
cpg_merged = merge_island_windows(cpg_raw, max_gap=50)
print("\n=== CpG Islands ===")
for s, e, gc, oe in cpg_merged:
    print(f"  Island at {s}-{e} ({e-s} bp), GC={gc:.2f}, CpG O/E={oe:.2f}")
    print(f"    Relative to TSS: {s-tss:+d} to {e-tss:+d}")

# 3. Find GATA motifs (example TFBS)
gata_hits = [(m.start(), m.group()) for m in re.finditer('AGATAA', test_promoter.upper())]
print("\n=== GATA Binding Sites ===")
for pos, motif in gata_hits:
    print(f"  Found '{motif}' at position {pos} (TSS{pos-tss:+d})")

In [ ]:
# Visualize the annotated promoter
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# GC content profile
win_size = 50
gc_profile = []
gc_pos = []
for i in range(0, len(test_promoter) - win_size, 5):
    win = test_promoter[i:i + win_size]
    gc_profile.append((win.count('G') + win.count('C')) / win_size)
    gc_pos.append(i + win_size // 2 - tss)

axes[0].plot(gc_pos, gc_profile, color='steelblue', linewidth=1)
axes[0].axhline(0.5, color='red', linestyle='--', alpha=0.5)
axes[0].set_ylabel('GC content')
axes[0].set_title('Comprehensive Promoter Analysis')

# CpG density profile
cpg_profile = []
cpg_pos = []
for i in range(0, len(test_promoter) - win_size, 5):
    win = test_promoter[i:i + win_size]
    cpg_count = win.count('CG')
    cpg_profile.append(cpg_count)
    cpg_pos.append(i + win_size // 2 - tss)

axes[1].plot(cpg_pos, cpg_profile, color='darkorange', linewidth=1)
axes[1].set_ylabel('CpG count per window')

# Mark features on all axes
for ax in axes:
    ax.axvline(0, color='red', linewidth=2, linestyle='--', alpha=0.7)
    for s, e, _, _ in cpg_merged:
        ax.axvspan(s - tss, e - tss, alpha=0.15, color='green')

# TATA and TFBS positions
tata_positions = [pos - tss for pos, _ in tata_hits]
gata_positions = [pos - tss for pos, _ in gata_hits]

axes[2].eventplot([tata_positions], lineoffsets=2, colors='orange', linewidths=3, label='TATA')
axes[2].eventplot([gata_positions], lineoffsets=1, colors='purple', linewidths=3, label='GATA TFBS')
axes[2].set_yticks([1, 2])
axes[2].set_yticklabels(['GATA', 'TATA'])
axes[2].set_ylabel('Motif hits')
axes[2].set_xlabel('Position relative to TSS (bp)')
axes[2].legend(loc='upper right')

# Add TSS label
axes[0].annotate('TSS', xy=(0, axes[0].get_ylim()[1]), fontsize=12, color='red',
                  ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---
## 8. Sequence Logos and Information Content

A **sequence logo** is a graphical representation of a motif where the height of each letter at each position is proportional to its information content (bits). The total height at position $i$ is:

$$R_i = \log_2(4) - H_i = 2 - \left(-\sum_{b} f_{b,i} \log_2 f_{b,i}\right)$$

where $H_i$ is the Shannon entropy at position $i$.

In [ ]:
def compute_information_content(pfm):
    """Compute information content at each position of a PFM."""
    length = len(pfm['A'])
    n_seqs = sum(pfm[b][0] for b in 'ACGT')
    ic = []
    
    for i in range(length):
        entropy = 0.0
        for b in 'ACGT':
            freq = pfm[b][i] / n_seqs
            if freq > 0:
                entropy -= freq * np.log2(freq)
        ic.append(2.0 - entropy)  # max entropy for DNA is 2 bits
    
    return ic

# Compute IC for our TATA box PFM
ic = compute_information_content(pfm)
n_seqs = len(tata_sites)

# Create a simple text-based logo visualization
fig, ax = plt.subplots(figsize=(10, 4))
positions_range = range(len(ic))

for i in positions_range:
    # Sort bases by frequency at this position
    base_freqs = [(b, pfm[b][i] / n_seqs) for b in 'ACGT']
    base_freqs.sort(key=lambda x: x[1])
    
    y_offset = 0
    colors = {'A': '#00CC00', 'C': '#0000CC', 'G': '#FFB300', 'T': '#CC0000'}
    for base, freq in base_freqs:
        height = freq * ic[i]
        if height > 0.01:
            ax.bar(i, height, bottom=y_offset, color=colors[base], width=0.8, edgecolor='white')
            if height > 0.15:
                ax.text(i, y_offset + height / 2, base, ha='center', va='center',
                        fontsize=14, fontweight='bold', color='white')
        y_offset += height

ax.set_xticks(list(positions_range))
ax.set_xticklabels([f'{i+1}' for i in positions_range])
ax.set_xlabel('Position in motif')
ax.set_ylabel('Information content (bits)')
ax.set_title('TATA Box Motif -- Information Content')
ax.set_ylim(0, 2.1)
plt.tight_layout()
plt.show()

print("Information content per position:")
for i, val in enumerate(ic):
    print(f"  Position {i+1}: {val:.3f} bits")

---
## 9. TFBS Enrichment Analysis

A common analysis is to test whether a particular TFBS is **enriched** in the promoters of a set of genes compared to background. This is directly relevant to understanding co-regulation.

In the rice promoter dataset, TFBS data was available from TRANSFAC matrices. Each gene had position-wise TFBS annotation, and informative TFs could be distinguished from general TFs.

In [ ]:
# Simulate TFBS enrichment data
np.random.seed(42)

# Simulate TFBS data for genes in two conditions: stress-responsive vs. housekeeping
n_stress = 100
n_house = 100
tf_names = ['ABI4', 'DREB1', 'MYB', 'WRKY', 'bZIP', 'NAC', 'ERF', 'HSF']

# Stress-responsive genes have more DREB1, WRKY, and HSF binding sites
stress_enriched = {'ABI4': 1.2, 'DREB1': 3.5, 'MYB': 1.5, 'WRKY': 4.0,
                   'bZIP': 1.1, 'NAC': 2.0, 'ERF': 2.5, 'HSF': 3.0}
baseline = {'ABI4': 1.0, 'DREB1': 1.0, 'MYB': 1.2, 'WRKY': 1.0,
            'bZIP': 1.3, 'NAC': 1.0, 'ERF': 1.0, 'HSF': 0.8}

data_rows = []
for tf in tf_names:
    stress_counts = np.random.poisson(stress_enriched[tf], n_stress)
    house_counts = np.random.poisson(baseline[tf], n_house)
    for c in stress_counts:
        data_rows.append({'TF': tf, 'group': 'Stress-responsive', 'count': c})
    for c in house_counts:
        data_rows.append({'TF': tf, 'group': 'Housekeeping', 'count': c})

tfbs_df = pd.DataFrame(data_rows)

# Visualize
fig, ax = plt.subplots(figsize=(12, 5))
pivot = tfbs_df.groupby(['TF', 'group'])['count'].mean().unstack()
pivot.plot(kind='bar', ax=ax, width=0.7)
ax.set_ylabel('Mean TFBS count per gene')
ax.set_title('TFBS Enrichment: Stress-Responsive vs. Housekeeping Genes')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Statistical test for enrichment (Mann-Whitney U)
from scipy import stats

print("TFBS Enrichment Test (Mann-Whitney U, stress vs. housekeeping)")
print("=" * 65)
print(f"{'TF':<10} {'Stress mean':>12} {'House mean':>12} {'U-stat':>10} {'p-value':>12} {'Sig?':>6}")
print("-" * 65)

for tf in tf_names:
    stress = tfbs_df[(tfbs_df['TF'] == tf) & (tfbs_df['group'] == 'Stress-responsive')]['count']
    house = tfbs_df[(tfbs_df['TF'] == tf) & (tfbs_df['group'] == 'Housekeeping')]['count']
    u_stat, p_val = stats.mannwhitneyu(stress, house, alternative='greater')
    sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'ns'
    print(f"{tf:<10} {stress.mean():>12.2f} {house.mean():>12.2f} {u_stat:>10.0f} {p_val:>12.2e} {sig:>6}")

---
## 10. Summary

In this notebook we covered:

1. **Gene regulation basics**: promoters, enhancers, transcription factors, TSS
2. **Promoter elements**: TATA box detection, CpG islands, TFBS identification
3. **CpG island detection**: sliding window algorithm with GC and O/E thresholds
4. **DNA methylation**: patterns around TSS and their role in gene regulation
5. **Sequence analysis**: GC content profiles, dinucleotide frequencies
6. **PWM construction**: building and scoring with position weight matrices
7. **TFBS enrichment**: comparing TFBS distributions between gene groups

These techniques form the foundation of promoter analysis in regulatory genomics.

---
## Exercises

### Exercise 1: Custom CpG Island Finder

Write a function `find_cpg_islands(sequence, min_length=300, gc_thresh=0.55, oe_thresh=0.65)` that:
1. Scans the sequence using a 200 bp sliding window (step=1)
2. Identifies all positions where both thresholds are met
3. Merges consecutive qualifying positions into islands
4. Returns only islands with total length >= `min_length`

Test it on the synthetic promoter from Section 4.

In [ ]:
# Your solution here


### Exercise 2: TATA Box Position Analysis

Using the `generate_promoter_collection()` function to create 500 promoters:
1. Search each promoter for TATA boxes (use the relaxed pattern `TATA[AT]A[AT]`)
2. Record positions of all TATA boxes relative to TSS (at position 1000)
3. Plot a histogram of TATA box positions
4. What fraction of TATA boxes fall in the expected -20 to -35 window?

In [ ]:
# Your solution here


### Exercise 3: Build a PWM for a Custom Motif

The E-box motif (CACGTG) is bound by bHLH transcription factors. Given these observed binding sites:

```
CACGTG, CACGTG, CACGTG, CACGTG, CACGTG,
CATGTG, CATGTG, CACGCG, CACGTG, CACATG,
CACGTG, CACGTG, CATGTG, CACGTG, CACGTG,
CACGAG, CACGTG, CACGTG, CACGTG, CATGTG
```

1. Build a PFM and convert it to a PWM
2. What is the information content at each position?
3. Which position is most conserved? Which is most variable?
4. Scan the `test_promoter` from Section 7 and report all hits with score > 3.0

In [ ]:
# Your solution here


### Exercise 4: Methylation and Expression Correlation

Simulate a dataset of 300 genes with:
- `promoter_methylation`: fraction of CpGs methylated in the promoter (0 to 1), drawn from `np.random.beta(2, 5)`
- `expression_level`: log2 expression, negatively correlated with methylation. Use: `expression = 10 - 8 * methylation + np.random.normal(0, 1.5)`

Tasks:
1. Create a scatter plot of methylation vs. expression
2. Compute the Pearson and Spearman correlations
3. Divide genes into 3 groups by methylation tertiles and compare expression using a box plot
4. Run a Kruskal-Wallis test to determine if the groups differ significantly

In [ ]:
# Your solution here


### Exercise 5: Promoter Feature Comparison

Write a function `analyze_promoter(sequence, tss_position)` that returns a dictionary with:
- `'gc_content'`: overall GC content
- `'cpg_islands'`: list of (start, end) tuples for detected CpG islands
- `'tata_boxes'`: list of positions (relative to TSS) of TATA boxes
- `'cpg_oe_ratio'`: overall CpG observed/expected ratio
- `'upstream_gc'`: GC content of the region -500 to -1 relative to TSS
- `'downstream_gc'`: GC content of the region +1 to +500 relative to TSS

Generate 100 promoters using `generate_promoter_collection()`, analyze each one, and create a summary DataFrame. Which features show the most variation across promoters?

In [ ]:
# Your solution here


---

[← Previous: Microbial Diversity Analysis](../../Tier_3_Applied_Bioinformatics/04_Microbial_Diversity/01_microbial_diversity.ipynb) | [Next: Statistics for Bioinformatics →](../../Tier_3_Applied_Bioinformatics/06_Statistics_for_Bioinformatics/01_statistics_for_bioinformatics.ipynb)